<a href="https://colab.research.google.com/github/shauryasachdev/Vizuara_CV/blob/main/VGG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from torchsummary import summary
import wandb
import pathlib
import numpy as np
import os
import time

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
time.sleep(10)

Mounted at /content/drive


In [3]:
print("\n=== 2. Checking your exact paths ===")
base_dir = '/content/drive/MyDrive/alexnet_flowers_dataset'
train_dir = f"{base_dir}/train"
val_dir   = f"{base_dir}/val"

print(f"Base folder exists?      {os.path.exists(base_dir)}")
print(f"train folder exists?     {os.path.exists(train_dir)}")
print(f"val folder exists?       {os.path.exists(val_dir)}")

if os.path.exists(base_dir):
    print("\nContents of your dataset folder:")
    print(os.listdir(base_dir))
else:
    print("\nMyDrive root contents (to help you spot the real location):")
    print(os.listdir('/content/drive/MyDrive/'))


=== 2. Checking your exact paths ===
Base folder exists?      True
train folder exists?     True
val folder exists?       True

Contents of your dataset folder:
['.DS_Store', 'train', 'val']


In [4]:
# ================================================
# Step 0: Initialize Wandb
# ================================================

wandb.init(project="vgg-flowers-v5", config={
    "epochs":50,
    "batch_size":16,
    "learning_rate":0.001,
    "architecture":"VGG",
    "pretrained":True,
    "input_size":224
})

# Shortcut to config values
config = wandb.config

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shauryasachdev to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
# ================================================
# Step 1: Data Preparation
# ================================================

# Transforms for training and validation
data_transforms = {
  'train': transforms.Compose([
      transforms.Resize((224,224)),
      transforms.RandomHorizontalFlip(),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406],
                          [0.229, 0.224, 0.225])
  ]),
  'val': transforms.Compose([
      transforms.Resize((224,224)),
      transforms.ToTensor(),
      transforms.Normalize([0.485, 0.456, 0.406],
                          [0.229, 0.224, 0.225])
  ])
}

# Your exact path as Path object
data_dir = pathlib.Path('/content/drive/MyDrive/alexnet_flowers_dataset/')
train_dir = pathlib.Path('/content/drive/MyDrive/alexnet_flowers_dataset/train')
val_dir = pathlib.Path('/content/drive/MyDrive/alexnet_flowers_dataset/val')

train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms['train'])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=config.batch_size)

In [6]:
# ================================================
# Step 2: Load Pretrained Model
# ================================================


model = models.vgg16(pretrained=config.pretrained)

# Freeze the feature layers
for param in model.features.parameters():
    param.requires_grad = False

# Replace the classifier with a fresh head for 5 classes
model.classifier = nn.Sequential(
    nn.Linear(25088, 4096),
    nn.ReLU(inplace=True),
    nn.Dropout(),
    nn.Linear(4096, 4096),
    nn.ReLU(inplace=True),
    nn.Dropout(0),
    nn.Linear(4096, 5)   # 5 flower classes dataset
)

# Only unfreeze the final classifier layer (index - 1)
for param in model.classifier[-1].parameters():
    param.requires_grad = True

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

wandb.watch(model, log="all", log_freq=10)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 89.0MB/s]


In [7]:
# ================================================
# Step 3: Loss and Optimizer
# ================================================

criterion = nn.CrossEntropyLoss()

# Lower LR + only optimize trainable parameters
optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

In [8]:
def train_model(model, criterion, optimizer, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_correct = 0
        train_total = 0
        running_loss = 0.0

        print(f"\nEpoch {epoch + 1}/{epochs}")
        print("-" * 30)

        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            batch_correct = (preds == labels).sum().item()
            train_correct += batch_correct
            train_total += labels.size(0)

            # Print every 10 batches
            if (i + 1) % 10 == 0:
                batch_acc = batch_correct / labels.size(0)
                print(f"[Batch {i+1}/{len(train_loader)}] Loss: {loss.item():.4f}, Batch Acc: {batch_acc:.4f}")

        train_acc = train_correct / train_total
        wandb.log({"epoch": epoch + 1, "train_loss": running_loss, "train_accuracy": train_acc})
        print(f"Epoch {epoch+1} Summary - Loss: {running_loss:.4f}, Train Accuracy: {train_acc:.4f}")

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        wandb.log({"epoch": epoch + 1, "val_accuracy": val_acc})
        print(f"Validation Accuracy: {val_acc:.4f}")


In [ ]:
# ===================
# Train the model
# ===================
train_model(model, criterion, optimizer, train_loader, val_loader, epochs=config.epochs)


Epoch 1/50
------------------------------
[Batch 10/251] Loss: 1.4733, Batch Acc: 0.5625
[Batch 20/251] Loss: 0.6050, Batch Acc: 0.7500
[Batch 30/251] Loss: 0.8837, Batch Acc: 0.6250
[Batch 40/251] Loss: 0.7186, Batch Acc: 0.6250
[Batch 50/251] Loss: 0.3937, Batch Acc: 0.9375
[Batch 60/251] Loss: 0.9470, Batch Acc: 0.6250
[Batch 70/251] Loss: 0.7764, Batch Acc: 0.7500
[Batch 80/251] Loss: 0.2307, Batch Acc: 0.9375
[Batch 90/251] Loss: 1.1291, Batch Acc: 0.7500
[Batch 100/251] Loss: 0.7843, Batch Acc: 0.7500
[Batch 110/251] Loss: 1.0246, Batch Acc: 0.4375
[Batch 120/251] Loss: 0.3405, Batch Acc: 0.8750
[Batch 130/251] Loss: 0.7805, Batch Acc: 0.6875
[Batch 140/251] Loss: 0.5421, Batch Acc: 0.8125
[Batch 150/251] Loss: 0.5691, Batch Acc: 0.8125
[Batch 160/251] Loss: 0.7920, Batch Acc: 0.7500
[Batch 170/251] Loss: 0.2374, Batch Acc: 0.9375
[Batch 180/251] Loss: 0.2043, Batch Acc: 0.9375
[Batch 190/251] Loss: 0.2476, Batch Acc: 0.9375
[Batch 200/251] Loss: 0.0417, Batch Acc: 1.0000
[Batch